# 03 — Quant Feature Engineering

1. Setup y reconstrucción del split temporal
2. Features de trayectoria
3. Features de volatilidad
4. Features de dinámica intradía
5. Dataset final de features

In [9]:
import pandas as pd
import numpy as np

input_train = pd.read_csv("../data/input_training.csv")
output_train = pd.read_csv("../data/output_training_gmEd6Zt.csv")

input_test = pd.read_csv("../data/input_test.csv")
output_test = pd.read_csv("../data/output_test_random.csv")

return_cols = [f"r{i}" for i in range(53)]

train = input_train.merge(
    output_train,
    on="ID",
    how="inner",
    validate="one_to_one"
)

In [10]:
# Reconstruimos exactamente el split decidido en 02:
# Split temporal definido en 02_validation_and_preprocessing
TRAIN_END_DAY = 401
VAL_START_DAY = 402

train_dev = train.loc[train["day"] <= TRAIN_END_DAY].copy()
val_dev = train.loc[train["day"] >= VAL_START_DAY].copy()
print("Train observations:", len(train_dev))
print("Validation observations:", len(val_dev))
print("Train day range:",train_dev["day"].min(),"-",train_dev["day"].max())
print("Validation day range:",val_dev["day"].min(),"-",val_dev["day"].max())

Train observations: 673751
Validation observations: 169548
Train day range: 0 - 401
Validation day range: 402 - 502


## 2. Features de trayectoria intradía

La exploración realizada en `01_research_hypotheses.ipynb` mostró que el retorno
acumulado contiene una relación no lineal con el régimen futuro: trayectorias con
desplazamientos intradía extremos presentan una mayor probabilidad de terminar en
un régimen direccional, mientras que trayectorias cercanas a cero presentan una
mayor frecuencia de la clase neutral.

También observamos que esta estructura aparece en distintas partes de la sesión,
por lo que resulta útil conservar información sobre cómo evoluciona el desplazamiento
a lo largo del día.

En esta sección transformamos la secuencia de 53 retornos en un conjunto compacto
de características de trayectoria.

Las features se calculan sobre los retornos originales expresados en basis points
para preservar su interpretación financiera.

### 2.1 Retorno acumulado

Para el *feature engineering* de horizontes intradía, consideraremos lo siguiente:

* **Tratamiento de NaN**: Calculamos el retorno usando únicamente valores observados (sin imputar ceros antes).
* **Control de Outliers**: Evitamos el producto compuesto y la suma de log-retornos, ya que existen valores anómalos inferiores al -100%.

Utilizaremos como aproximación de desplazamiento (*path*):

$$R_{\text{path}} \approx \sum_{t=0}^{52} r_t$$


In [11]:
def row_sum_observed(df, cols):
    return df[cols].sum(axis=1, skipna=True)


for df in [train_dev, val_dev]:
    df["path_return_bps"] = row_sum_observed(df, return_cols)

#2.2 Trayectoria por ventanas
#Recuperamos la división que utilizamos en H2:
early_cols = [f"r{i}" for i in range(0, 18)]
middle_cols = [f"r{i}" for i in range(18, 36)]
late_cols = [f"r{i}" for i in range(36, 53)]
for df in [train_dev, val_dev]:

    df["return_early_bps"] = row_sum_observed(df, early_cols)
    df["return_middle_bps"] = row_sum_observed(df, middle_cols)
    df["return_late_bps"] = row_sum_observed(df, late_cols)

### 2.2 Segmentación de la trayectoria

* **Qué se hace**: Calculamos retornos acumulados por tramos del día: $R_{\text{early}}$, $R_{\text{middle}}$ y $R_{\text{late}}$.
* **Por qué**: Permite al modelo diferenciar la dirección del camino. Por ejemplo, distingue $(+, +, +)$ de $(+, +, -)$, aunque ambos terminen con un desplazamiento final similar.

### 2.3 Magnitud del desplazamiento

* **Qué se hace**: Incluimos explícitamente el valor absoluto del desplazamiento total: $|R_{\text{path}}|$.
* **Por qué**: Captura relaciones no lineales. Los extremos de los retornos acumulados (grandes subidas o grandes caídas) presentan mayor probabilidad de activar un régimen direccional.


In [12]:
for df in [train_dev, val_dev]:
    df["abs_path_return_bps"] = df["path_return_bps"].abs()
#Esto separa dos conceptos:

#path_return_bps      → dirección del desplazamiento
#abs_path_return_bps  → intensidad del desplazamiento

### 2.4 Cambio entre inicio y final de la trayectoria

* **Qué se hace**: Calculamos una única variable de evolución temporal: 
$$\Delta R = R_{\text{late}} - R_{\text{early}}$$
* **Por qué**: Sustituye y simplifica diez variables por una sola, capturando de forma directa el cambio neto en la tendencia intradía.


In [13]:
for df in [train_dev, val_dev]:
    df["early_late_change_bps"] = (
        df["return_late_bps"]
        - df["return_early_bps"]
    )
path_features = [
    "path_return_bps",
    "abs_path_return_bps",
    "return_early_bps",
    "return_middle_bps",
    "return_late_bps",
    "early_late_change_bps",
]

train_dev[path_features].describe().T

,count,mean,std,min,25%,50%,75%,max
path_return_bps,673751.0,1047.193482,544780.771046,-1.987691e+04,-107.73,-6.21,79.69,4.310702e+08
abs_path_return_bps,673751.0,1249.431252,544780.344760,0.000000e+00,38.72,92.98,192.90,4.310702e+08
return_early_bps,673751.0,1050.800577,544780.719419,-1.989236e+04,-81.57,-1.93,61.67,4.310702e+08
return_middle_bps,673751.0,-2.344122,101.645075,-4.096710e+03,-41.14,0.00,37.65,6.699440e+03
return_late_bps,673751.0,-1.262972,79.871266,-3.140540e+03,-30.60,0.00,28.76,5.230770e+03
early_late_change_bps,673751.0,-1052.063549,544780.742871,-4.310702e+08,-70.38,2.63,89.10,1.990870e+04


In [14]:
#2.1 Crear versión robusta de los retornos originales
#Usaría 0.1%–99.9% por intervalo. Es suficientemente conservador y ya vimos esos cuantiles durante el diagnóstico.
# Límites estimados exclusivamente sobre train
lower_bounds = train_dev[return_cols].quantile(0.001)
upper_bounds = train_dev[return_cols].quantile(0.999)
def clip_raw_returns(df, return_cols, lower, upper):
    out = df.copy()

    out[return_cols] = out[return_cols].clip(
        lower=lower,
        upper=upper,
        axis=1
    )

    return out


train_feat = clip_raw_returns(
    train_dev,
    return_cols,
    lower_bounds,
    upper_bounds
)

val_feat = clip_raw_returns(
    val_dev,
    return_cols,
    lower_bounds,
    upper_bounds
)
# calculamos las mismas 6 features
def add_path_features(df):

    df = df.copy()

    df["path_return_bps"] = df[return_cols].sum(axis=1, skipna=True)

    df["return_early_bps"] = df[early_cols].sum(axis=1, skipna=True)
    df["return_middle_bps"] = df[middle_cols].sum(axis=1, skipna=True)
    df["return_late_bps"] = df[late_cols].sum(axis=1, skipna=True)

    df["abs_path_return_bps"] = df["path_return_bps"].abs()

    df["early_late_change_bps"] = (
        df["return_late_bps"]
        - df["return_early_bps"]
    )

    return df


train_feat = add_path_features(train_feat)
val_feat = add_path_features(val_feat)
train_feat[path_features].describe().T

,count,mean,std,min,25%,50%,75%,max
path_return_bps,673751.0,-33.468035,292.186858,-3491.96809,-107.52,-6.33,79.35,4185.64313
abs_path_return_bps,673751.0,164.155974,244.020587,0.00000,38.64,92.70,191.23,4185.64313
return_early_bps,673751.0,-29.811015,265.006347,-3437.91677,-81.54,-1.97,61.53,3000.45530
return_middle_bps,673751.0,-2.373711,92.840922,-1552.37245,-41.10,0.00,37.57,2040.33730
return_late_bps,673751.0,-1.283308,71.961493,-1124.35735,-30.57,0.00,28.70,1546.85292
early_late_change_bps,673751.0,28.527707,273.466402,-3097.84761,-70.16,2.68,88.85,3571.48609


### Conclusiones clave del análisis

* **Eliminación de ruido**: Desapareció la contaminación numérica; los extremos ahora están en una escala económicamente interpretable.
* **Estructura de volatilidad**: El inicio de la sesión concentra la mayor dispersión, decreciendo a lo largo del día:
$$\sigma_{\text{early}} = 265 > \sigma_{\text{middle}} = 93 > \sigma_{\text{late}} = 72$$
* **Asimetría del desplazamiento**: El sesgo a la baja ocurre exclusivamente al principio de la sesión, facilitando la detección de *momentum*, reversión o neutralidad:
$$\text{median}(R_{\text{path}}) = -6.33 \quad \text{vs.} \quad \text{median}(R_{\text{early}}) \approx -1.97, \; \text{median}(R_{\text{middle}}) = 0, \; \text{median}(R_{\text{late}}) = 0$$


### 3.0 Definir los retornos que alimentan las features
Si el DataFrame donde ya aplicaste imputación + robust scaling + clipping es X_train_baseline, entonces:


## 3. Volatilidad y actividad de la trayectoria

Las features anteriores resumen el desplazamiento direccional de la trayectoria. Sin embargo, dos trayectorias con un retorno acumulado similar pueden haber seguido dinámicas intradía muy diferentes.

Por ello incorporamos medidas de actividad independientes de la dirección:

- **Realized volatility:** magnitud cuadrática de los movimientos intradía.
- **Mean absolute return:** magnitud típica de los movimientos de 5 minutos.
- **Early volatility share:** proporción de la actividad total concentrada en la primera parte de la sesión.

Estas variables buscan capturar no sólo cuánto se desplazó el precio, sino también cómo se distribuyó la actividad durante la ventana observada.

In [18]:
# 3.1 realized_vol_bps
# Como nuestros r0,...,r52 ya están expresados en bps:

# Realized volatility de la trayectoria
train_feat["realized_vol_bps"] = np.sqrt(
    (train_feat[return_cols] ** 2).sum(axis=1, skipna=True)
)

val_feat["realized_vol_bps"] = np.sqrt(
    (val_feat[return_cols] ** 2).sum(axis=1, skipna=True)
)
train_feat["realized_vol_bps"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    673751.000000
mean        194.379075
std         216.936893
min           0.000000
50%         140.393075
75%         219.068160
90%         350.012985
95%         481.243293
99%        1333.768890
max        2627.706886
Name: realized_vol_bps, dtype: float64

### Actividad de la trayectoria

* **Fórmula**: 
$$RV_i = \sqrt{\sum_{t=1}^{53} r_{i,t}^2}$$
* **Concepto**: No representa volatilidad anualizada; funciona estrictamente como una métrica de actividad realizada dentro de la trayectoria observada.

### Volatilidad ajustada

* **Fórmula**:
$$RV_i^{\text{adj}} = \sqrt{\frac{53}{n_i} \sum_{t \in \text{observed}} r_{i,t}^2}$$
* **Concepto**: Escala la actividad realizada multiplicando por la proporción de datos faltantes, corrigiendo el sesgo cuando la trayectoria tiene datos omitidos ($n_i$ representa el número de retornos observados).




In [19]:
#Ajustar la volatilidad por disponibilidad
#Como todas las trayectorias deberían representar 53 intervalos,
# podemos calcular primero el número de retornos observados:
train_feat["n_obs"] = train_feat[return_cols].notna().sum(axis=1)
val_feat["n_obs"] = val_feat[return_cols].notna().sum(axis=1)
train_feat["realized_vol_adj_bps"] = np.where(train_feat["n_obs"] > 0,np.sqrt(
    (53 / train_feat["n_obs"])* (train_feat[return_cols] ** 2).sum(axis=1, skipna=True)),np.nan)

val_feat["realized_vol_adj_bps"] = np.where(val_feat["n_obs"] > 0,np.sqrt(
        (53 / val_feat["n_obs"])* (val_feat[return_cols] ** 2).sum(axis=1, skipna=True)),
    np.nan)
#Esto no reconstruye los retornos faltantes. Simplemente hace comparable la escala de actividad 
# entre trayectorias con diferente número de observaciones, bajo una aproximación de que los retornos 
# disponibles son representativos de la intensidad de la trayectoria.
train_feat[["n_obs", "realized_vol_bps", "realized_vol_adj_bps"]].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])

,n_obs,realized_vol_bps,realized_vol_adj_bps
count,673751.000000,673751.000000,672072.000000
mean,47.289200,194.379075,213.834448
std,12.236509,216.936893,251.879709
min,0.000000,0.000000,0.000000
50%,53.000000,140.393075,148.376782
75%,53.000000,219.068160,236.782387
90%,53.000000,350.012985,394.379383
95%,53.000000,481.243293,551.631053
99%,53.000000,1333.768890,1514.187265
max,53.000000,2627.706886,14307.745568


### Validación en Holdout

* **Antecedente**: En la sección *01* confirmamos que una mayor volatilidad está fuertemente asociada con la probabilidad del régimen direccional:
$$P(|r_{\text{eod}}| = 1)$$
* **Objetivo**: Comprobar si esta relación se mantiene en el *holdout* temporal utilizando la variable ya limpia y ajustada ($RV_i^{\text{adj}}$).


In [ ]:
train_feat["directional"] = (train_feat["reod"].abs() == 1).astype(int)
val_feat["directional"] = (val_feat["reod"].abs() == 1).astype(int)
q20, q40, q60, q80 = train_feat["realized_vol_adj_bps"].quantile([0.20, 0.40, 0.60, 0.80])
vol_bins = [
    -np.inf,
    q20,
    q40,
    q60,
    q80,
    np.inf
]

vol_labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]

train_feat["vol_quintile"] = pd.cut(
    train_feat["realized_vol_adj_bps"],
    bins=vol_bins,
    labels=vol_labels
)

val_feat["vol_quintile"] = pd.cut(
    val_feat["realized_vol_adj_bps"],
    bins=vol_bins,
    labels=vol_labels
)

In [22]:
vol_signal_check = pd.DataFrame({
    "train": (
        train_feat
        .groupby("vol_quintile", observed=True)["directional"]
        .mean()
    ),
    "validation": (
        val_feat
        .groupby("vol_quintile", observed=True)["directional"]
        .mean()
    )
})

vol_signal_check

,train,validation
vol_quintile,,
Q1,0.405952,0.380460
Q2,0.577053,0.560826
Q3,0.644613,0.633511
Q4,0.688098,0.679802
Q5,0.643775,0.636724
